In [ ]:
source("helpers/init.r")

source("helpers/documentation.r")


In [ ]:
cohorts    <- fread(paste0(READY_DATA, "cohorts.txt"))
plt_cohort <- cohorts %>% filter(!ref) %>% pull(cohort) %>% unique()
ref_cohort <- cohorts %>% filter(ref)  %>% pull(cohort) %>% unique()

# 0 - Load all figures

In [ ]:
setwd(READY_FIG)
clinical   <- readRDS("clinical.rds")
signatures <- readRDS("signatures.rds")
markers    <- readRDS("markers.rds")
drivers    <- readRDS("drivers.rds")
actionable <- readRDS("actionable.rds")
coverage   <- readRDS("coverage.rds")
schema     <- readRDS("schemas.rds")$schema
circos     <- ggit("circos.png")
doids      <- if ("doids" %in% colnames(cohorts))
  cohorts %>% filter(cohort == plt_cohort) %>% pull(doids) %>% unique() %>%
  { .[!is.na(.) & trimws(.) != ""] } else character(0)
time_stamp <- runQuery("markers", cols = "`pipeline_date`", sampleIds = head(cohorts$sampleId, 1)) %>% pull(pipeline_date)

# 1 - Theme helpers

In [ ]:
set_panel_title <- function(size = 22, hjust = 0, vjust = 1.5, face = "bold") {
  theme(plot.title = element_text(hjust = hjust, vjust = vjust, size = size,
                                  colour = hartwig_red, face = face))
}
marg  <- function(t = 0, r = 0, b = 0, l = 0) {
  theme(plot.margin = margin(t = t, r = r, b = b, l = l, "cm"))
}
panel <- theme(panel.background = element_rect(colour = "black", linewidth = 4, fill = NA))
ts    <- set_title(size = 14, face = "plain", color = "black")


make_section_header <- function(title, size = 22) {
  ggdraw() +
    draw_label(title, x = 0.01, hjust = 0, fontface = "bold",
               color = hartwig_red, size = size)
}


# 2 - Left panel

In [ ]:
clinical_panel <-
  clinical$figure +
  labs(title = "Cohort Metadata", subtitle = paste0(db_icon(), " Clinical")) +
  set_panel_title() + blue_subtitle

# ── Assemble markers panel with section headers ────────────────────────────────
row1 <- markers$tumor_char
row2 <- plot_grid(markers$genome, markers$driver, markers$telomere,
                  nrow = 1, rel_widths = c(2.5, 2.5, 1.5))
row3 <- plot_grid(markers$mut_burden, markers$immune,
                  nrow = 1, rel_widths = c(5, 2))
row4 <- markers$sv

marker_rows    <- list(make_section_header("Tumor Characteristics"), row1,
                       make_section_header("Genomic Landscape"), row2, row3, row4)
marker_heights <- c(0.3, 2, 0.3, 2, 2, 2)

if (!is.null(markers$rna)) {
  marker_rows    <- c(marker_rows, list(make_section_header("Transcriptomic Landscape"), markers$rna))
  marker_heights <- c(marker_heights, 0.3, 2)
}

markers_figure <- plot_grid(plotlist = marker_rows, ncol = 1, rel_heights = marker_heights)

# Clinical panel height matches one data row in markers (Tumor Char / Transcriptomic)
clinical_height <- 2 / sum(marker_heights) * 12

left <- go(arrangeGrob(
  clinical_panel   + marg(r = 0, b = .6),
  markers_figure   + marg(r = .2, b = .2),
  heights = c(clinical_height, 12)
))


# 3 - Middle panel

In [ ]:
middle <- go(arrangeGrob(
  signatures$signatures +
    labs(title = "Processes Underlying Mutations", subtitle = paste0(tool_icon(), " SIGS")) +
    set_panel_title(hjust = .5, vjust = 1.5) + blue_subtitle + theme(plot.subtitle = element_markdown(hjust = 0.5)) + marg(r = -2, b = -1, l = -2),
  schema + theme(plot.margin = margin(.2, .2, .6, .2, "cm")),
  markers$cuppa_stacked +
    labs(title = "Primary Tumor Prediction", subtitle = paste0(tool_icon(), " CUPPA")) +
    set_panel_title(hjust = .5, vjust = 1.5) + blue_subtitle + theme(plot.subtitle = element_markdown(hjust = 0.5)),
  circos +
    labs(title = "Copy Number Alteration Profile", subtitle = paste0(tool_icon(), " PURPLE  ·  COBALT")) +
    set_panel_title(hjust = .5, vjust = 1.5) + blue_subtitle + theme(plot.subtitle = element_markdown(hjust = 0.5)) + marg(b = -.7, l = -.7),
  layout_matrix = cbind(c(1,2,3,4)), heights = c(3.0, 4.0, 1.5, 4.2)
))


# 4 - Right panel

In [ ]:
widths    <- c(1.91, 5)
onco_pads <- marg(t = .05, r = .1, b = .3, l = .1)

germline_panel <-
  plot_grid(drivers$labels$GERMLINE, drivers$onco$germline,
            align = "h", axis = "tb", rel_widths = widths, ncol = 2, nrow = 1) +
  labs(title = "Germline Predisposition", subtitle = paste0(tool_icon(), " PURPLE  ·  LINX")) +
  set_panel_title() + blue_subtitle + onco_pads

driver_panel <-
  plot_grid(drivers$labels$SOMATIC, drivers$onco$somatic,
            align = "h", axis = "tb", rel_widths = widths, ncol = 2, nrow = 1) +
  ggtitle("Cancer Driver Landscape") +
  labs(subtitle = paste0(tool_icon(), " PURPLE  ·  LINX  ·  VirusInterpreter")) +
  blue_subtitle +
  set_panel_title() + onco_pads

actionable_panel <-
  plot_grid(actionable$labels, actionable$onco + theme(legend.text = element_text(size = 9.5)),
            align = "h", axis = "tb", rel_widths = c(1.615, 5.285), ncol = 2, nrow = 1) +
  ggtitle("Potentially Actionable Events") +
  labs(subtitle = paste0(db_icon(), " CKB")) +
  blue_subtitle +
  set_panel_title() + onco_pads

right <- as_ggplot(arrangeGrob(
  driver_panel, actionable_panel, germline_panel,
  layout_matrix = cbind(c(1,1,1,1,1,1,1,2,2,2,2,2,2,3,3,3))
))

# 5 - Coverage panel

In [ ]:
leg <- get_legend(
  coverage$bto +
    guides(fill = guide_legend(nrow = 2, byrow = TRUE, reverse = TRUE)) +
    theme(legend.spacing.x = unit(.5, "cm"), legend.text = element_text(size = 11.5))
)

cov_panel <-
  plot_grid(
    leg,
    coverage$bto + theme(legend.position = "none", axis.title.y = element_blank(),
                         plot.title = element_text(hjust = .5, size = 14)),
    coverage$avg + theme(axis.title.y = element_blank(),
                         plot.title = element_text(hjust = .5, size = 14)),
    coverage$tso + theme(axis.title.y = element_blank(),
                         plot.title = element_text(hjust = .45, size = 16)),
    coverage$amp + theme(axis.title.y = element_blank(),
                         plot.title = element_text(hjust = .45, size = 16)),
    align = "v", axis = "b", rel_heights = c(1.1, 5, 5, 7, 7), ncol = 1, nrow = 5
  ) +
  labs(title = "WGS vs Panel Coverage", subtitle = paste0(db_icon(), " CKB")) +
  set_panel_title(hjust = .7) + blue_subtitle + theme(plot.subtitle = element_markdown(hjust = 0.5, size = 11, color = "#304b99", face = "bold", margin = margin(b = 4))) + onco_pads

# 6 - Assemble vignette

In [ ]:
vignette <- as_ggplot(arrangeGrob(
  left, middle, right, cov_panel,
  layout_matrix = t(c(rep(1,12), rep(2,9), rep(3,13), rep(4,8)))
))

raw_title <- cohorts %>% filter(!ref) %>% head(1) %>% pull(title)
title <- paste0("The Genomic and Actionability Landscape of ", title_display(raw_title))

image <- vignette + marg(t = 3)

# 7 - Add logo and annotations

In [ ]:
logo    <- readPNG(paste0(LOGO_DIR, "Hartwig_logo_2025.png"))
cc      <- readPNG(paste0(LOGO_DIR, "license.png"))
license <- "CC BY-NC: This license enables reusers to distribute, remix, adapt,\nand build upon the material in any medium or format for noncommercial\npurposes only, and only so long as attribution is given to the creator."
website <- "Website: https://www.hartwigmedicalfoundation.nl\nTools: https://github.com/hartwigmedical/hmftools\nDatabase: https://www.hartwigmedicalfoundation.nl/en/data/database/"

image_go <-
  image +
  annotation_custom(rasterGrob(logo), xmin = 0.00, xmax = 0.08, ymin = 1.005, ymax = 1.085) +
  annotation_custom(rasterGrob(cc),   xmin = .95, xmax = 1,   ymin = 1.02, ymax = 1.08) +
  annotate("text", x = .972, y = 1.02, label = license, size = 1.3) +
  annotate("text", x = .874, y = 1.038, label = website, size = 3.2)

# 8 - Annotate and save

In [ ]:
# 8a - Build annotated figure
oncotree_codes <- tryCatch({
  codes <- fread(paste0(READY_DATA, "oncotree_codes.txt"))$oncotree
  message("✓ Oncotree codes loaded: ", paste(codes, collapse = ", "))
  codes
}, error = function(e) {
  message("⚠ oncotree_codes.txt not found or unreadable: ", conditionMessage(e))
  character(0)
})
oncotree_str <- if (length(oncotree_codes) > 0)
  paste0("  |  Oncotree codes: ", paste(oncotree_codes, collapse = ", ")) else ""
overlap    <- length(intersect(
  cohorts %>% filter(!ref) %>% pull(sampleId),
  cohorts %>% filter(ref)  %>% pull(sampleId)
))
excl_note  <- if (overlap == 0) "  |  Cohorts: mutually exclusive" else
              paste0("  |  Cohorts: ", overlap, " shared sample(s)")
extra <- paste0("DOIDs: ", paste(doids, collapse = ", "), oncotree_str, "  |  Date: ", time_stamp, excl_note)
message("Bottom label: ", extra)
url   <- "Documentation: https://www.hartwigmedicalfoundation.nl/en/data/vignettes/"

# add_sub carves out a bottom strip for the tool legend
go_bottom <- ggdraw(add_sub(image_go, paste0(strrep("\n", BOTTOM_STRIP_LINES)),
                            x = 0, hjust = 0, size = 9, fontface = "plain"))

options(repr.plot.width = VIGNETTE_WIDTH, repr.plot.height = VIGNETTE_HEIGHT, repr.plot.res = 100)
lets_go <-
  go_bottom +
    draw_label(title, x = 0.43, y = 0.979, hjust = 0.5, vjust = 0.5,
               size = 33, fontface = "bold", color = "black") +
    draw_grob(
      grid::roundrectGrob(
        x = 0.5, y = 0.5, width = unit(1, "npc"), height = unit(1, "npc"),
        r = unit(6, "pt"),
        gp = grid::gpar(fill = "#fcfcfc", col = "#cccccc", lwd = 0.8)
      ),
      x = -0.01, y = 0.001, width = 1.02, height = 0.081,
      hjust = 0, vjust = 0
    ) +
    draw_label("Documentation", x = -0.01, y = 0.084, hjust = 0, vjust = 0,
               size = 8, color = "#aaaaaa", fontface = "italic") +
    annotate("richtext", x = 0/5,  y = 0.075, label = doc_tools_c1, size = 3.9,
             hjust = 0, vjust = 1, fill = NA, label.color = NA) +
    annotate("richtext", x = 1/5,  y = 0.075, label = doc_tools_c2, size = 3.9,
             hjust = 0, vjust = 1, fill = NA, label.color = NA) +
    annotate("richtext", x = 2/5,  y = 0.075, label = doc_tools_c3, size = 3.9,
             hjust = 0, vjust = 1, fill = NA, label.color = NA) +
    annotate("richtext", x = 3/5,  y = 0.075, label = doc_tools_c4, size = 3.9,
             hjust = 0, vjust = 1, fill = NA, label.color = NA) +
    annotate("richtext", x = 4/5,  y = 0.075, label = doc_tools_c5, size = 3.9,
             hjust = 0, vjust = 1, fill = NA, label.color = NA) +
    annotate("richtext", x = 0/5,  y = 0.033, label = doc_row2_c1, size = 3.9,
             hjust = 0, vjust = 1, fill = NA, label.color = NA) +
    annotate("richtext", x = 1/5,  y = 0.033, label = doc_row2_c2, size = 3.9,
             hjust = 0, vjust = 1, fill = NA, label.color = NA) +
    annotate("richtext", x = 2/5,  y = 0.033, label = doc_row2_c3, size = 3.9,
             hjust = 0, vjust = 1, fill = NA, label.color = NA) +
    annotate("richtext", x = 3/5,  y = 0.033, label = doc_row2_c4, size = 3.9,
             hjust = 0, vjust = 1, fill = NA, label.color = NA) +
    annotate("richtext", x = 4/5,  y = 0.033, label = doc_row2_c5, size = 3.9,
             hjust = 0, vjust = 1, fill = NA, label.color = NA) +
    annotate("text", x = 0, y = 0.008, label = extra, size = 3.2,
             hjust = 0, vjust = 1, color = "#333333") +
    annotate("text", x = 1, y = 0.008, label = url,   size = 3.2,
             hjust = 1, vjust = 1, color = "#304b99", fontface = "italic") +
    marg(r = 2, l = 2, t = 1, b = 1)
invisible(lets_go)

In [ ]:
# 8b - Preview
lets_go

In [ ]:
# 8c - Save (layout F)
run_mode    <- tryCatch(trimws(readLines(paste0(READY_DATA, "run_mode.txt"))[1]),
                        error = function(e) "overlap")
mode_suffix <- if (identical(run_mode, "exclusive")) "_excl" else "_overlap"
ref_name <- cohorts %>% filter(ref) %>% pull(cohort) %>% unique() %>% head(1)
o_file <- clean_filename(paste0(cohorts %>% filter(!ref) %>% head(1) %>% pull(cohort),
                                "_vignette", mode_suffix, "_", ref_name, ".png"))

# ── Output folder: one subfolder per oncotree_code (groups siblings together) ────
oncotree_code <- if ("oncotree_code" %in% colnames(cohorts))
  cohorts %>% filter(!ref) %>% pull(oncotree_code) %>% unique() %>% .[!is.na(.) & trimws(.) != ""] else character(0)
out_dir <- if (length(oncotree_code) == 1)
  paste0(SHARE_FIG, oncotree_code, "/") else SHARE_FIG
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
ggsave(paste0(out_dir, o_file), plot = lets_go, width = VIGNETTE_WIDTH, height = VIGNETTE_HEIGHT, dpi = VIGNETTE_DPI)
if (length(oncotree_code) == 1) {
  message("✓ Saved: ", oncotree_code, "/", o_file)
} else {
  message("✓ Saved: ", o_file, " (no oncotree_code — placed in share root)")
}
# ── Multi-size export (--sizes flag) ────────────────────────────────────────────────────
multi_size <- tryCatch(trimws(readLines(paste0(READY_DATA, "multi_size.txt"))[1]),
                       error = function(e) "false")
if (identical(multi_size, "true")) {
  if (!requireNamespace("magick", quietly = TRUE))
    stop("Package 'magick' is required for --sizes export. Install with: install.packages('magick')")
  library(magick)
  full_path <- paste0(out_dir, o_file)
  base_name <- sub("\\.png$", "", o_file)
  img <- image_read(full_path)
  # Thumb: 400px wide
  thumb_file <- paste0(out_dir, base_name, "_thumb.png")
  image_write(image_scale(img, "400"), thumb_file)
  message("✓ Thumb saved:   ", basename(thumb_file))
  # Preview: 2400px wide
  preview_file <- paste0(out_dir, base_name, "_preview.png")
  image_write(image_scale(img, "2400"), preview_file)
  message("✓ Preview saved: ", basename(preview_file))
}